# 05 - CNN on Mel-spectrogram (Colab T4 GPU)

- **輸入**：Mel-spectrogram `.npy` (128, 94) → 增加 channel → (1, 128, 94)
- **模型**：輕量 CNN（4 block conv + GAP），定義在 `src/models.py`
- **評估**：StratifiedGroupKFold (K=5, group=speaker_id)
- **訓練**：Adam + ReduceLROnPlateau + Early Stopping (patience=7)
- **精度**：fp16 mixed precision

In [ ]:
# === 安裝相依套件 ===
!pip install -q plotly kaleido

In [ ]:
# === 路徑設定（支援 Colab 與本地執行）===
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SER-Project')
except (ImportError, ModuleNotFoundError):
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

sys.path.append(str(PROJECT_ROOT / 'src'))

MELSPEC_DIR = PROJECT_ROOT / 'data' / 'features' / 'melspec'
CKPT_DIR = PROJECT_ROOT / 'models' / 'checkpoints' / 'cnn'
RESULTS_DIR = PROJECT_ROOT / 'results'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
import torch
print(f'Device: {"GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "CPU"}')

In [ ]:
# === Imports ===
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

from models import SERConvNet, EarlyStopping

# === 超參數 ===
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
PATIENCE = 7
N_SPLITS = 5
RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === 可重現性 ===
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'Device: {DEVICE}')

In [ ]:
# === Dataset ===
class MelSpecDataset(Dataset):
    """Mel-spectrogram dataset: 載入 .npy 並增加 channel 維度。"""

    def __init__(self, paths: list[str], labels: np.ndarray):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        mel = np.load(self.paths[idx])              # (128, 94)
        mel = mel[np.newaxis, :, :]                  # (1, 128, 94)
        x = torch.tensor(mel, dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


# === 載入 metadata ===
df = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
le = LabelEncoder()
y_all = le.fit_transform(df['emotion'].values)
groups = df['speaker_id'].values
classes = le.classes_.tolist()

# 建立完整路徑清單
mel_paths = [str(PROJECT_ROOT / p) for p in df['melspec_path']]

print(f'Samples: {len(df):,}, Classes: {classes}')
print(f'Speakers: {df["speaker_id"].nunique()}')

In [ ]:
# === 訓練 / 評估函式 ===

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(X)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        with torch.cuda.amp.autocast():
            logits = model(X)
            loss = criterion(logits, y)
        total_loss += loss.item() * len(y)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    n = len(all_labels)
    preds = np.array(all_preds)
    labels = np.array(all_labels)
    return (
        total_loss / n,
        accuracy_score(labels, preds),
        f1_score(labels, preds, average='weighted'),
        f1_score(labels, preds, average='macro'),
        preds,
        labels,
    )

In [ ]:
# === 5-Fold 訓練 ===
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
all_results = {'folds': [], 'confusion_matrices': [], 'classification_reports': [], 'classes': classes}

for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(mel_paths, y_all, groups)):
    print(f'\n{"="*60}')
    print(f'  Fold {fold_idx + 1}/{N_SPLITS}')
    print(f'{"="*60}')

    # 資料集
    train_paths = [mel_paths[i] for i in train_idx]
    test_paths = [mel_paths[i] for i in test_idx]
    train_ds = MelSpecDataset(train_paths, y_all[train_idx])
    test_ds = MelSpecDataset(test_paths, y_all[test_idx])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Class weight（根據訓練集類別分布計算逆頻率權重）
    cw = compute_class_weight('balanced', classes=np.arange(len(classes)), y=y_all[train_idx])
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
    print(f'  Class weights: {dict(zip(classes, [f"{w:.3f}" for w in cw]))}')

    # 模型
    model = SERConvNet(num_classes=len(classes)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler()
    early_stopping = EarlyStopping(patience=PATIENCE)

    best_val_loss = float('inf')
    t_start = time.time()

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
        val_loss, val_acc, val_f1w, val_f1m, _, _ = evaluate(model, test_loader, criterion, DEVICE)
        scheduler.step(val_loss)
        early_stopping(val_loss)

        # 儲存最佳 checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), CKPT_DIR / f'cnn_fold{fold_idx+1}_best.pt')

        lr_now = optimizer.param_groups[0]['lr']
        if (epoch + 1) % 5 == 0 or early_stopping.should_stop:
            print(f'  Epoch {epoch+1:>3d}: train_loss={train_loss:.4f}, '
                  f'val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, '
                  f'F1(w)={val_f1w:.4f}, F1(m)={val_f1m:.4f}, lr={lr_now:.1e}')

        if early_stopping.should_stop:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    # 載入最佳權重做最終評估
    model.load_state_dict(torch.load(CKPT_DIR / f'cnn_fold{fold_idx+1}_best.pt', weights_only=True))
    val_loss, val_acc, val_f1w, val_f1m, preds, labels = evaluate(model, test_loader, criterion, DEVICE)
    elapsed = time.time() - t_start

    cm = confusion_matrix(labels, preds).tolist()
    report = classification_report(labels, preds, target_names=classes, output_dict=True)

    fold_result = {
        'fold': fold_idx + 1,
        'accuracy': val_acc,
        'f1_weighted': val_f1w,
        'f1_macro': val_f1m,
        'best_val_loss': best_val_loss,
        'stopped_epoch': epoch + 1,
        'time_sec': round(elapsed, 1),
        'test_samples': len(test_idx),
        'train_samples': len(train_idx),
    }
    all_results['folds'].append(fold_result)
    all_results['confusion_matrices'].append(cm)
    all_results['classification_reports'].append(report)

    print(f'\n  [Fold {fold_idx+1} Best] acc={val_acc:.4f}, F1(w)={val_f1w:.4f}, '
          f'F1(macro)={val_f1m:.4f}, time={elapsed:.0f}s')

    # 釋放 GPU 記憶體
    del model, optimizer, scaler, criterion
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('  All folds done!')
print(f'{"="*60}')

In [ ]:
# === 結果匯總 + 儲存 ===
metrics_df = pd.DataFrame(all_results['folds'])

summary = {
    'accuracy_mean': metrics_df['accuracy'].mean(),
    'accuracy_std': metrics_df['accuracy'].std(),
    'f1_weighted_mean': metrics_df['f1_weighted'].mean(),
    'f1_weighted_std': metrics_df['f1_weighted'].std(),
    'f1_macro_mean': metrics_df['f1_macro'].mean(),
    'f1_macro_std': metrics_df['f1_macro'].std(),
}
all_results['summary'] = summary

# 儲存 JSON
with open(RESULTS_DIR / 'cnn_results.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print('CNN 5-Fold Results:')
print(f'  Accuracy:    {summary["accuracy_mean"]:.4f} +/- {summary["accuracy_std"]:.4f}')
print(f'  F1 (weighted): {summary["f1_weighted_mean"]:.4f} +/- {summary["f1_weighted_std"]:.4f}')
print(f'  F1 (macro):    {summary["f1_macro_mean"]:.4f} +/- {summary["f1_macro_std"]:.4f}')
print(f'\nResults saved to {RESULTS_DIR / "cnn_results.json"}')
print(f'Checkpoints saved to {CKPT_DIR}')

In [ ]:
# === Per-fold 表格 ===
metrics_df[['fold', 'accuracy', 'f1_weighted', 'f1_macro', 'stopped_epoch', 'time_sec']]